# 01 · India-focused synthetic AIS demo (Milestone 1)

**Pilot region:** Gulf of Mannar – Palk Bay, south Tamil Nadu coast (`IN-GOM`).

This notebook runs from a clean runtime on **CPU only** (no GPU cell in this milestone; ~1–2 min):

1. generates a clearly labelled **synthetic** India-coast AIS dataset (Mode A);
2. normalises it to the canonical schema and **anonymises** vessel identifiers;
3. cleans it (duplicates, invalid coordinates, impossible speeds, AIS sentinels);
4. computes movement and India-specific geospatial features;
5. detects **explicit AIS-gap events** with reception and boundary context;
6. validates gap detection against hidden simulation truth;
7. plots trajectories on an India-focused map (static + interactive);
8. builds leakage-safe train/validation/test splits;
9. saves processed Parquet files, figures and metrics.

> ⚠️ **Synthetic demo data.** Every vessel, track and label here is simulated (`synthetic_demo_*`). Nothing in this notebook is evidence about real vessels or real illegal fishing. Boundaries and protected-area outlines are approximate placeholders. Outputs describe *suspicious behaviour / investigation priority*, never illegality.

In [ ]:
# ---- Setup: works in Google Colab (CPU is enough) and locally --------------------
import os, sys, subprocess, pathlib, zipfile, glob
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Dan-droid-A-113/IUU-Fishing-GNN-India"   # clones the code; set to "" to upload the zip instead
USE_DRIVE = False      # True = keep project + outputs on Google Drive (restart-safe)

if IN_COLAB:
    PROJECT_DIR = pathlib.Path("/content/india-fishing-gnn")
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = pathlib.Path("/content/drive/MyDrive/india-fishing-gnn")
    if not (PROJECT_DIR / "src").exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(PROJECT_DIR)], check=True)
        else:
            # accepts renamed downloads such as "india-fishing-gnn (1).zip"
            zips = sorted(glob.glob("/content/india-fishing-gnn*.zip"), key=os.path.getmtime)
            if not zips:
                from google.colab import files
                print("Project zip not found - choose india-fishing-gnn.zip from your computer:")
                uploaded = files.upload()   # file picker appears below this cell
                zips = [os.path.abspath(n) for n in uploaded if n.endswith(".zip")]
            if not zips:
                raise FileNotFoundError("No project zip uploaded.")
            print("Extracting", zips[-1])
            zipfile.ZipFile(zips[-1]).extractall(PROJECT_DIR.parent)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_DIR / "requirements-colab.txt")], check=True)
else:
    PROJECT_DIR = pathlib.Path.cwd()
    while not (PROJECT_DIR / "src").exists() and PROJECT_DIR != PROJECT_DIR.parent:
        PROJECT_DIR = PROJECT_DIR.parent

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Project directory:", PROJECT_DIR, "| Colab:", IN_COLAB)

## 0 · Configuration

In [ ]:
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore", category=FutureWarning)

from src.config import (SyntheticConfig, PreprocessConfig, SYNTHETIC_DIR, PROCESSED_DIR,
                        FIGURE_DIR, METRIC_DIR, REGIONS, SYNTHETIC_CAVEAT, SEED)
from src.india_layers import get_layers
from src.synthetic import generate_synthetic_dataset
from src.data_io import normalize_schema, anonymize_ids, save_table
from src.preprocessing import run_preprocessing, resample_tracks
from src.graph_builder import proximity_pairs, gap_prism_pairs
from src.splits import (vessel_disjoint_split, time_disjoint_split, region_disjoint_split,
                        segment_table, check_no_leakage)
from src.evaluation import attach_gap_truth, planted_dark_windows, gap_detection_report
from src import visualization as viz

np.random.seed(SEED)
for d in (SYNTHETIC_DIR, PROCESSED_DIR, FIGURE_DIR, METRIC_DIR):
    d.mkdir(parents=True, exist_ok=True)

FORCE_REGENERATE = False          # True = rebuild synthetic data even if cached
SYN_CFG = SyntheticConfig()       # 7 days, 50 vessels, seed 42
PRE_CFG = PreprocessConfig()      # gap threshold 60 min, max speed 40 kn, ...
print(SYNTHETIC_CAVEAT)
pd.DataFrame([vars(r) for r in REGIONS.values()])

## 1 · Generate (or load cached) synthetic AIS — Mode A

The simulator moves vessels minute by minute around real coastline geometry (offline land mask) and then *observes* them through a coverage model: synthetic coastal receivers, weak satellite pick-up for Class B fishing transponders, one scheduled receiver-cluster outage, transmitter glitches, intentional blackouts, and AIS switched off in port. Observable AIS and **hidden truth** are written to separate files; hidden truth is used only for evaluation.

In [ ]:
raw_path = SYNTHETIC_DIR / "ais_raw_synthetic.parquet"
if raw_path.exists() and not FORCE_REGENERATE:
    tables = {"ais_raw": pd.read_parquet(raw_path),
              "registry": pd.read_parquet(SYNTHETIC_DIR / "vessel_registry_synthetic.parquet")}
    for k in ("truth_reports", "events", "vessel_truth", "injected_errors"):
        tables[k] = pd.read_parquet(SYNTHETIC_DIR / "hidden_truth" / f"{k}.parquet")
    print("Loaded cached synthetic dataset")
else:
    tables = generate_synthetic_dataset(SYN_CFG)
    print("Generated and saved synthetic dataset")

raw, registry, events = tables["ais_raw"], tables["registry"], tables["events"]
print(f"raw AIS rows: {len(raw):,} | vessels: {raw.mmsi.nunique()} | "
      f"time span: {raw.timestamp.min()} -> {raw.timestamp.max()}")
display(registry.vessel_class.value_counts().rename("vessels").to_frame())
display(raw.head())

In [ ]:
# Planted scenarios (hidden truth - evaluation only)
display(events.groupby(["event_type", "synthetic_demo_label"]).size().rename("events").to_frame())
display(tables["truth_reports"].miss_reason.value_counts(normalize=True).round(3)
        .rename("share of scheduled AIS reports").to_frame())

## 2 · Canonical schema and anonymisation

Raw identifiers are replaced by keyed-hash IDs (`vessel_xxxxxxxx`). The mapping is kept in `data/processed/_private/` (git-ignored). For real data set a private salt in the `AIS_ID_SALT` environment variable.

In [ ]:
canon = normalize_schema(raw, source="synthetic_demo", registry=registry)
ais_anon, id_map = anonymize_ids(canon)
save_table(id_map, PROCESSED_DIR / "_private" / "id_map_synthetic")
assert not ais_anon.astype(str).apply(lambda c: c.str.contains("SYNTH")).any().any()
display(ais_anon.head())

## 3 · Cleaning, kinematics, geospatial features, AIS-gap events

In [ ]:
layers = get_layers("IN-GOM")
display(layers.metadata_table()[["name", "kind", "crs", "source", "uncertainty"]])

res = run_preprocessing(ais_anon, layers, PRE_CFG)
ais, gaps, report = res["ais"], res["gaps"], res["report"]
display(pd.Series(report, name="value").to_frame())

In [ ]:
# Did cleaning catch the injected data-quality errors?
inj = tables["injected_errors"].error_type.value_counts()
check = pd.DataFrame({
    "injected": inj,
    "handled": pd.Series({"duplicate": report["duplicates_removed"],
                          "invalid_coordinate": report["invalid_coordinates_removed"],
                          "position_jump": report["impossible_speed_flagged"] + report["outside_regions_removed"],
                          "sog_not_available": report["sog_not_available"]})})
display(check)

In [ ]:
cols = ["vessel_id", "timestamp", "latitude", "longitude", "sog", "dt_minutes", "distance_m",
        "turn_rate", "fishing_motion_score", "mpa_distance_m", "eez_distance_m", "port_distance_m",
        "ais_gap_id", "gap_duration_min", "sar_detection_flag", "segment_id"]
display(ais[cols].sample(5, random_state=1))
gap_cols = ["ais_gap_id", "vessel_id", "vessel_type", "gap_start", "gap_duration_min", "displacement_m",
            "gap_context", "concurrent_silence_frac", "n_neighbours_before", "crossed_boundary",
            "min_mpa_distance_m", "pre_gap_sog_mean"]
display(gaps[gaps.gap_context == "at_sea"][gap_cols].head(10))

## 4 · Validate gap detection against hidden truth (synthetic only)

Two things must hold: every planted blackout longer than the threshold is detected, **and** most detected gaps are benign (port stays, coverage, receiver outage, transmitter faults). The second point is why an AIS gap alone must never be treated as a suspicion label.

In [ ]:
gaps_t = attach_gap_truth(gaps, tables["truth_reports"], id_map)
dark_w = planted_dark_windows(tables["truth_reports"], id_map)
gap_metrics = gap_detection_report(gaps_t, dark_w, PRE_CFG.gap_threshold_min)
print(json.dumps(gap_metrics, indent=2))
display(pd.crosstab(gaps_t.truth_cause, gaps_t.gap_context, margins=True))
within = gaps_t[gaps_t.truth_cause != "between_voyages"]
display(within.groupby("truth_cause").agg(
    n=("gap_duration_min", "size"), median_duration_min=("gap_duration_min", "median"),
    mean_concurrent_silence=("concurrent_silence_frac", "mean"),
    median_displacement_km=("displacement_m", lambda s: s.median() / 1000)).round(2))
fig = viz.plot_gap_diagnostics(gaps_t, PRE_CFG.gap_threshold_min, FIGURE_DIR / "m1_gap_diagnostics.png")
plt.show()

## 5 · Maps

In [ ]:
fig = viz.plot_tracks_map(ais, layers, gaps,
                          title="Synthetic AIS tracks - Gulf of Mannar / Palk Bay pilot (7 days)",
                          path=FIGURE_DIR / "m1_tracks_map.png")
plt.show()

In [ ]:
# Zoom on one planted blackout: what the pipeline sees vs. hidden truth
ex = gaps_t[gaps_t.truth_cause == "dark_intentional"].sort_values("gap_duration_min").iloc[-1]
raw_id = id_map.set_index("vessel_id").raw_id[ex.vessel_id]
tr = tables["truth_reports"]
true_path = tr[(tr.raw_id == raw_id) & (tr.timestamp >= ex.gap_start) & (tr.timestamp <= ex.gap_end)]
fig = viz.plot_gap_example(ais, ex, layers, true_path, path=FIGURE_DIR / "m1_planted_blackout_example.png",
                           title=f"Planted synthetic blackout - {ex.vessel_id} ({ex.vessel_type})")
plt.show()

In [ ]:
m = viz.folium_map(ais, layers, gaps, path=FIGURE_DIR / "m1_tracks_map.html")
m  # interactive; background tiles need internet

In [ ]:
# Feature sanity check: do kinematic features behave as expected per simulated behaviour?
obs_truth = tr[tr.observed].merge(id_map, on="raw_id")[["vessel_id", "timestamp", "behavior"]]
# align timestamp precision (pandas 2 vs 3 may differ between the two tables)
ais_ts = ais.assign(timestamp=ais.timestamp.astype("datetime64[ns, UTC]"))
obs_truth = obs_truth.assign(timestamp=obs_truth.timestamp.astype("datetime64[ns, UTC]"))
joined = pd.merge_asof(ais_ts.sort_values("timestamp"), obs_truth.sort_values("timestamp"),
                       on="timestamp", by="vessel_id", tolerance=pd.Timedelta("2s"), direction="nearest")
fig = viz.plot_behaviour_features(joined, FIGURE_DIR / "m1_behaviour_features.png")
plt.show()
display(joined.groupby("behavior")[["sog", "fishing_motion_score"]].median().round(2))

## 6 · Optional resampling (never across long gaps)

In [ ]:
rs = resample_tracks(ais, PRE_CFG)
print(f"{len(rs):,} resampled rows at {PRE_CFG.resample_min}-min; "
      f"max bracket span = {rs.bracket_span_min.max():.1f} min (limit {PRE_CFG.max_interp_gap_min})")
assert (rs.bracket_span_min <= PRE_CFG.max_interp_gap_min).all()

## 7 · Leakage-safe splits

* **Vessel-disjoint**: vessels are grouped so an interaction never straddles splits. Two link types: (a) ≥ 40 min within 500 m of each other outside ports (BallTree search, not all-pairs); (b) *possible dark encounters* — a slow vessel that the silent vessel could have reached during its AIS gap (space-time prism test). Link (b) is what keeps blackout rendezvous partners together.
* **Time-disjoint**: days 1–4 train, day 5 validation, days 6–7 test, with a 6 h embargo; voyage segments that touch a boundary are dropped, never cut.
* **Region-disjoint**: only `IN-GOM` exists so far; the function is ready for when the Gujarat–Maharashtra or Odisha–West Bengal regions are added.

In [ ]:
pairs = proximity_pairs(ais, radius_m=500, bin_min=10, min_bins=4, exclude_port_m=3000)
prism = gap_prism_pairs(ais, gaps, vmax_kn=10, max_partner_sog_kn=2, min_bins=3)
print(f"{len(pairs)} sustained close-proximity pairs | {len(prism)} possible-dark-encounter links")
links = list(zip(pairs.vessel_a, pairs.vessel_b)) + list(zip(prism.vessel_a, prism.vessel_b))

# stratify by whether a vessel has any planted suspicious report (labels used ONLY to balance splits)
sus_raw = set(tr.loc[tr.suspicious, "raw_id"])
strata = {v: int(r in sus_raw) for r, v in zip(id_map.raw_id, id_map.vessel_id)}
vsplit = vessel_disjoint_split(ais.vessel_id.unique(), links, strata, seed=SEED)
ais["split_vessel"] = ais.vessel_id.map(vsplit)

seg = segment_table(ais)
t0 = ais.timestamp.min().floor("D")
tsplit = time_disjoint_split(seg, [t0 + pd.Timedelta(days=4), t0 + pd.Timedelta(days=5)], pd.Timedelta("6h"))
ais["split_time"] = ais.segment_id.map(tsplit)
rsplit = region_disjoint_split(ais.region_id.unique(), test_regions=[])

# planted rendezvous partners must end up in the same vessel split
rdv = events[events.event_type == "low_speed_rendezvous"]
to_v = dict(zip(id_map.raw_id, id_map.vessel_id))
same = [len({vsplit.get(to_v.get(x)) for x in e.split(",")}) == 1 for e in rdv.vessels]
split_summary = {
    "vessel_split_counts": pd.Series(vsplit).value_counts().to_dict(),
    "suspicious_vessels_per_split": pd.Series({v: vsplit[v] for v, s in strata.items() if s}).value_counts().to_dict(),
    "time_split_segments": tsplit.value_counts().to_dict(),
    "region_split": rsplit,
    "rendezvous_pairs_kept_together": f"{sum(same)}/{len(same)}",
    "vessel_split_leakage_check": check_no_leakage(ais, "split_vessel"),
    "time_split_segment_check": check_no_leakage(ais, "split_time")["segments_unique"],
}
print(json.dumps(split_summary, indent=2, default=str))

## 8 · Save processed data and metrics

In [ ]:
p1 = save_table(ais, PROCESSED_DIR / "ais_processed_synthetic")
p2 = save_table(gaps, PROCESSED_DIR / "ais_gaps_synthetic")
p3 = save_table(pairs, PROCESSED_DIR / "proximity_pairs_synthetic")
save_table(prism, PROCESSED_DIR / "dark_encounter_links_synthetic")
summary = {
    "milestone": 1, "data_mode": "A_synthetic", "caveat": SYNTHETIC_CAVEAT, "region": "IN-GOM",
    "n_raw_rows": int(len(raw)), "n_clean_rows": int(len(ais)), "n_vessels": int(ais.vessel_id.nunique()),
    "vessel_hours_observed": float(ais.groupby("segment_id").timestamp.agg(lambda s: (s.max() - s.min()).total_seconds() / 3600).sum()),
    "cleaning": report, "gap_detection": gap_metrics, "splits": split_summary,
    "preprocess_config": PRE_CFG.to_dict(), "synthetic_config": SYN_CFG.to_dict(),
}
(METRIC_DIR / "m1_synthetic_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("saved:", p1, p2, p3, METRIC_DIR / "m1_synthetic_summary.json", sep="\n  ")
print("figures:", *sorted(p.name for p in FIGURE_DIR.glob("m1_*")), sep="\n  ")

## What Milestone 1 can and cannot conclude

**Can conclude**
* The pipeline runs end-to-end from a clean runtime: synthetic generation → schema → anonymisation → cleaning → kinematic and India-specific geospatial features → explicit AIS-gap events → maps → leakage-safe splits → saved artefacts.
* Cleaning catches the injected data errors, and the gap detector recovers **every planted blackout** longer than the threshold.
* Most detected gaps are **benign** (port stays, coverage, receiver outage, transmitter faults), so gap duration alone is not a usable suspicion signal. Context features such as concurrent silence, boundary crossing and displacement are needed.

**Cannot conclude**
* Nothing about real vessels, real fishing, or real illegal activity in Indian waters. All data are synthetic.
* Nothing about detection performance on real AIS. The coverage model, behaviours and label rates are assumptions.
* The MPA, maritime-boundary and fishing-ground layers are approximate placeholders and must be replaced by authoritative layers (WDPA, Marine Regions) before any real-data run.
* No model has been trained yet. Risk scores, graph learning and explanations start in Milestones 2–4.

**Next:** Milestone 2, deep-learning baselines (rule reference, MLP, GRU, TCN) trained on these splits.